# Formula 1 — All Seasons Data Pipeline (OpenF1 → Airtable)

This notebook generalizes the single-season `2024_Formula_1_Season_Data.ipynb` /
`2025_Formula_1_Season_Data.ipynb` notebooks into one pipeline that:

1. Fetches Formula 1 data for **every season available** from the
   [OpenF1 API](https://openf1.org/) (not just one hardcoded year).
2. Cleans/shapes it into tidy tables.
3. Runs the same kind of exploratory SQL joins as the originals, but across all years.
4. Pushes (upserts) the data into an **Airtable base** built to store it long-term.

**Airtable base created for this pipeline:** `Formula 1 Multi-Year Data`
(base id `appmZPMPRAjoKM5He`) with 7 tables: `Meetings`, `Sessions`, `Drivers`,
`Session Results`, `Starting Grid`, `Laps`, `Pit Stops`.

> Run the cells top-to-bottom. Steps 4–6 talk to the OpenF1 API (no key needed).
> Step 9 talks to Airtable and needs a Personal Access Token — see Step 2.


## 1. Pre-Requisites

In [ ]:
# If running in a fresh environment, uncomment to install dependencies:
# %pip install pandas pandasql requests

import json
import math
import os
import time
from datetime import datetime, timezone

import pandas as pd
import requests
from pandasql import sqldf


## 2. Configuration

`YEARS_TO_INCLUDE` controls the scope of the pull:

- `None` → keep **every** season the OpenF1 API returns (fully "all years").
- A list like `[2023, 2024, 2025]` → restrict to those seasons only.

Laps (and pit stops) are the highest-volume tables — a full multi-year pull can be
tens/hundreds of thousands of rows. `PUSH_LAPS_TO_AIRTABLE` / `PUSH_PITS_TO_AIRTABLE`
let you keep them in the notebook's dataframes (e.g. to export as CSV/Parquet) without
pushing that volume into Airtable, if your Airtable plan's record limits are a concern.


In [ ]:
# --- Scope ---
YEARS_TO_INCLUDE = None          # e.g. [2023, 2024, 2025] to restrict; None = all seasons available

# --- OpenF1 API ---
OPENF1_BASE_URL = "https://api.openf1.org/v1"

# --- Airtable ---
AIRTABLE_BASE_ID = "appmZPMPRAjoKM5He"   # "Formula 1 Multi-Year Data" base
AIRTABLE_TABLES = {
    "meetings": "Meetings",
    "sessions": "Sessions",
    "drivers": "Drivers",
    "session_results": "Session Results",
    "starting_grid": "Starting Grid",
    "laps": "Laps",
    "pit_stops": "Pit Stops",
}
AIRTABLE_PRIMARY_FIELD = {
    "meetings": "Meeting Key",
    "sessions": "Session Key",
    "drivers": "Driver Session Key",
    "session_results": "Result Key",
    "starting_grid": "Grid Key",
    "laps": "Lap Key",
    "pit_stops": "Pit Key",
}

PUSH_TO_AIRTABLE = True
PUSH_LAPS_TO_AIRTABLE = True
PUSH_PITS_TO_AIRTABLE = True
AIRTABLE_BATCH_SIZE = 10   # Airtable's per-request record limit for create/update/upsert


In [ ]:
# Airtable Personal Access Token (scopes needed: data.records:read, data.records:write,
# against the "Formula 1 Multi-Year Data" base). Prefer an environment variable; falls back
# to a hidden prompt so the token is never printed or committed to the notebook.
import getpass

AIRTABLE_API_KEY = os.environ.get("AIRTABLE_API_KEY")
if not AIRTABLE_API_KEY and PUSH_TO_AIRTABLE:
    AIRTABLE_API_KEY = getpass.getpass("Enter your Airtable Personal Access Token: ")


## 3. Helper Functions

In [ ]:
def fetch_data_to_dataframe(url: str, df_name: str = "data", columns_to_omit: list = None,
                             verbose: bool = True) -> pd.DataFrame:
    """Fetch a JSON endpoint into a DataFrame. Returns an empty DataFrame on any failure."""
    if verbose:
        print(f"Attempting to fetch {df_name} from: {url}")
    try:
        response = requests.get(url, timeout=60)
        response.raise_for_status()
        fetched_data = response.json()

        if fetched_data:
            df = pd.DataFrame(fetched_data)
            if verbose:
                print(f"Successfully fetched {len(df)} rows for {df_name}.")
            if columns_to_omit:
                existing_columns_to_drop = [c for c in columns_to_omit if c in df.columns]
                if existing_columns_to_drop:
                    df = df.drop(columns=existing_columns_to_drop)
            return df
        else:
            if verbose:
                print(f"No data found for {df_name} from the provided URL.")
            return pd.DataFrame()
    except requests.exceptions.RequestException as e:
        print(f"Error fetching {df_name} from {url}: {e}")
        return pd.DataFrame()


In [ ]:
def format_lap_duration(seconds):
    """Convert a lap duration in seconds (float) to an M:SS.mmm string."""
    if seconds is None or (isinstance(seconds, float) and math.isnan(seconds)):
        return None

    minutes, sec = divmod(seconds, 60)
    sec_int = int(sec)
    milliseconds = int(round((sec - sec_int) * 1000))

    return f"{int(minutes)}:{sec_int:02d}.{milliseconds:03d}"


# Run ad-hoc SQL against the dataframes in this notebook's global scope
pysqldf = lambda q: sqldf(q, globals())


def clean_records(df: pd.DataFrame) -> list:
    """Convert a DataFrame to a list of Airtable-ready record dicts (NaN -> dropped, numpy -> python)."""
    records = json.loads(df.to_json(orient="records", date_format="iso"))
    cleaned = []
    for rec in records:
        cleaned.append({k: v for k, v in rec.items() if v is not None})
    return cleaned


def chunked(seq, size):
    for i in range(0, len(seq), size):
        yield seq[i:i + size]


In [ ]:
def push_dataframe_to_airtable(df: pd.DataFrame, table_key: str, max_retries: int = 5) -> int:
    """Upsert a DataFrame into its mapped Airtable table, keyed on that table's primary field.

    Uses Airtable's upsert API (PATCH with performUpsert) so re-running the notebook is
    idempotent: existing rows are updated in place, new rows are created, keyed on the
    composite/natural key we built for each table.
    """
    if df.empty:
        print(f"Skipping {table_key}: no rows to push.")
        return 0

    table_name = AIRTABLE_TABLES[table_key]
    primary_field = AIRTABLE_PRIMARY_FIELD[table_key]
    url = f"https://api.airtable.com/v0/{AIRTABLE_BASE_ID}/{requests.utils.quote(table_name)}"
    headers = {
        "Authorization": f"Bearer {AIRTABLE_API_KEY}",
        "Content-Type": "application/json",
    }

    records = clean_records(df)
    pushed = 0

    for batch in chunked(records, AIRTABLE_BATCH_SIZE):
        payload = {
            "performUpsert": {"fieldsToMergeOn": [primary_field]},
            "records": [{"fields": rec} for rec in batch],
        }

        attempt = 0
        while True:
            resp = requests.patch(url, headers=headers, json=payload, timeout=60)
            if resp.status_code == 429:
                # Rate limited — back off and retry
                attempt += 1
                if attempt > max_retries:
                    resp.raise_for_status()
                time.sleep(2 ** attempt)
                continue
            if not resp.ok:
                print(f"Airtable error on {table_name}: {resp.status_code} {resp.text[:500]}")
            resp.raise_for_status()
            break

        pushed += len(batch)

    print(f"Pushed {pushed} rows into '{table_name}'.")
    return pushed


## 4. Fetch Core Reference Data (all seasons, one call each)

The OpenF1 endpoints below return their **entire history** when called without a
year/date filter — not just the current season — so each of these only needs to be
fetched once, regardless of how many seasons we ultimately keep.


In [ ]:
sessions_url        = f"{OPENF1_BASE_URL}/sessions"
meetings_url         = f"{OPENF1_BASE_URL}/meetings"
drivers_url          = f"{OPENF1_BASE_URL}/drivers"
session_results_url  = f"{OPENF1_BASE_URL}/session_result"
starting_grid_url    = f"{OPENF1_BASE_URL}/starting_grid"
pits_url             = f"{OPENF1_BASE_URL}/pit"

sessions_df         = fetch_data_to_dataframe(sessions_url, df_name="sessions data (all years)")
meetings_df         = fetch_data_to_dataframe(meetings_url, df_name="meetings data (all years)")
drivers_df          = fetch_data_to_dataframe(drivers_url, df_name="drivers data (all years)")
session_results_df  = fetch_data_to_dataframe(session_results_url, df_name="session result data (all years)", verbose=False)
starting_grid_df    = fetch_data_to_dataframe(starting_grid_url, df_name="starting grid data (all years)", verbose=False)
pits_df             = fetch_data_to_dataframe(pits_url, df_name="pit stop data (all years)", verbose=False)

print("\nAvailable seasons in sessions data:", sorted(sessions_df["year"].dropna().unique().tolist()))


## 5. Scope to Selected Years (optional filter)

Everything downstream is derived from `sessions_df`'s `year` column, so this is the
single place that enforces `YEARS_TO_INCLUDE`.


In [ ]:
if YEARS_TO_INCLUDE:
    sessions_scoped = sessions_df[sessions_df["year"].isin(YEARS_TO_INCLUDE)].copy()
else:
    sessions_scoped = sessions_df.copy()

race_sessions_all = sessions_scoped[sessions_scoped["session_type"] == "Race"].copy()
race_sessions      = race_sessions_all  # kept for parity with the single-season notebooks' naming

scoped_meeting_keys  = set(sessions_scoped["meeting_key"].dropna().unique())
scoped_session_keys  = set(sessions_scoped["session_key"].dropna().unique())

meetings_scoped        = meetings_df[meetings_df["meeting_key"].isin(scoped_meeting_keys)].copy()
drivers_scoped         = drivers_df[drivers_df["session_key"].isin(scoped_session_keys)].copy()
session_results_scoped = session_results_df[session_results_df["session_key"].isin(scoped_session_keys)].copy()
starting_grid_scoped   = starting_grid_df[starting_grid_df["session_key"].isin(scoped_session_keys)].copy()
pits_scoped            = pits_df[pits_df["session_key"].isin(scoped_session_keys)].copy() if not pits_df.empty else pits_df

print(f"Sessions in scope: {len(sessions_scoped)}")
print(f"Race sessions in scope: {len(race_sessions_all)}")
print(f"Meetings in scope: {len(meetings_scoped)}")
print(f"Drivers rows in scope: {len(drivers_scoped)}")
print(f"Session results in scope: {len(session_results_scoped)}")
print(f"Starting grid rows in scope: {len(starting_grid_scoped)}")
print(f"Pit stop rows in scope: {len(pits_scoped)}")


## 6. Fetch Laps Data Per Race Session (looped, all selected years)

Laps are too high-volume for a single unfiltered call, so — exactly like the original
notebooks — we loop `session_key` by `session_key` over every race (incl. sprint) session
currently in scope, across every season.


In [ ]:
all_laps = []
columns_to_exclude = ["segments_sector_1", "segments_sector_2", "segments_sector_3"]

for i, (idx, row) in enumerate(race_sessions_all.iterrows(), start=1):
    session_key = row["session_key"]
    laps_url = f"{OPENF1_BASE_URL}/laps?session_key={session_key}"

    laps = fetch_data_to_dataframe(laps_url, df_name=f"laps in session {session_key}",
                                    columns_to_omit=columns_to_exclude, verbose=False)

    if laps is not None and not laps.empty:
        laps["session_key"] = session_key
        laps["meeting_key"] = row["meeting_key"]
        laps["year"] = row["year"]
        all_laps.append(laps)

    if i % 25 == 0 or i == len(race_sessions_all):
        print(f"Fetched laps for {i}/{len(race_sessions_all)} race sessions...")

if all_laps:
    laps_df = pd.concat(all_laps, ignore_index=True)
    laps_df["lap_duration_formatted"] = laps_df["lap_duration"].apply(format_lap_duration)
    print(f"\nTotal laps collected across all seasons in scope: {len(laps_df)}")
else:
    laps_df = pd.DataFrame()
    print("No laps data found for the selected scope.")


## 7. Shape Data for Airtable

Build the composite/natural primary keys each Airtable table expects, rename columns to
match the Airtable field names created for this base, and normalize types (booleans,
NaN → omitted, dates → ISO strings).


In [ ]:
# --- Meetings ---
meetings_airtable = meetings_scoped.rename(columns={
    "meeting_key": "Meeting Key",
    "meeting_name": "Meeting Name",
    "meeting_official_name": "Meeting Official Name",
    "location": "Location",
    "country_name": "Country Name",
    "country_code": "Country Code",
    "circuit_short_name": "Circuit Short Name",
    "circuit_key": "Circuit Key",
    "date_start": "Date Start",
    "year": "Year",
})
meetings_airtable = meetings_airtable[[c for c in [
    "Meeting Key", "Meeting Name", "Meeting Official Name", "Location", "Country Name",
    "Country Code", "Circuit Short Name", "Circuit Key", "Date Start", "Year",
] if c in meetings_airtable.columns]]

# --- Sessions ---
sessions_airtable = sessions_scoped.rename(columns={
    "session_key": "Session Key",
    "meeting_key": "Meeting Key",
    "session_name": "Session Name",
    "session_type": "Session Type",
    "location": "Location",
    "country_name": "Country Name",
    "circuit_short_name": "Circuit Short Name",
    "date_start": "Date Start",
    "date_end": "Date End",
    "year": "Year",
})
sessions_airtable = sessions_airtable[[c for c in [
    "Session Key", "Meeting Key", "Session Name", "Session Type", "Location",
    "Country Name", "Circuit Short Name", "Date Start", "Date End", "Year",
] if c in sessions_airtable.columns]]

# --- Drivers (dedup one row per session+driver) ---
drivers_airtable = drivers_scoped.copy()
drivers_airtable["Driver Session Key"] = (
    drivers_airtable["session_key"].astype(str) + "_" + drivers_airtable["driver_number"].astype(str)
)
drivers_airtable = drivers_airtable.drop_duplicates(subset=["Driver Session Key"])
drivers_airtable = drivers_airtable.rename(columns={
    "driver_number": "Driver Number",
    "session_key": "Session Key",
    "meeting_key": "Meeting Key",
    "full_name": "Full Name",
    "name_acronym": "Name Acronym",
    "team_name": "Team Name",
    "team_colour": "Team Colour",
    "country_code": "Country Code",
})
sessions_year_lookup = sessions_scoped.set_index("session_key")["year"]
drivers_airtable["Year"] = drivers_airtable["Session Key"].map(sessions_year_lookup)
drivers_airtable = drivers_airtable[[c for c in [
    "Driver Session Key", "Driver Number", "Session Key", "Meeting Key", "Full Name",
    "Name Acronym", "Team Name", "Team Colour", "Country Code", "Year",
] if c in drivers_airtable.columns]]

# --- Session Results ---
results_airtable = session_results_scoped.copy()
results_airtable["Result Key"] = (
    results_airtable["session_key"].astype(str) + "_" + results_airtable["driver_number"].astype(str)
)
results_airtable = results_airtable.rename(columns={
    "session_key": "Session Key",
    "meeting_key": "Meeting Key",
    "driver_number": "Driver Number",
    "position": "Position",
    "number_of_laps": "Number Of Laps",
    "points": "Points",
    "dnf": "DNF",
    "dns": "DNS",
    "dsq": "DSQ",
    "duration": "Duration",
    "gap_to_leader": "Gap To Leader",
})
results_airtable["Gap To Leader"] = results_airtable["Gap To Leader"].astype(str)
results_airtable["Year"] = results_airtable["Session Key"].map(sessions_year_lookup)
results_airtable = results_airtable[[c for c in [
    "Result Key", "Session Key", "Meeting Key", "Driver Number", "Position",
    "Number Of Laps", "Points", "DNF", "DNS", "DSQ", "Duration", "Gap To Leader", "Year",
] if c in results_airtable.columns]]

# --- Starting Grid ---
grid_airtable = starting_grid_scoped.copy()
grid_airtable["Grid Key"] = (
    grid_airtable["session_key"].astype(str) + "_" + grid_airtable["driver_number"].astype(str)
)
grid_airtable = grid_airtable.rename(columns={
    "session_key": "Session Key",
    "meeting_key": "Meeting Key",
    "driver_number": "Driver Number",
    "position": "Position",
    "lap_duration": "Lap Duration",
})
grid_airtable["Year"] = grid_airtable["Session Key"].map(sessions_year_lookup)
grid_airtable = grid_airtable[[c for c in [
    "Grid Key", "Session Key", "Meeting Key", "Driver Number", "Position", "Lap Duration", "Year",
] if c in grid_airtable.columns]]

# --- Laps ---
if not laps_df.empty:
    laps_airtable = laps_df.copy()
    laps_airtable["Lap Key"] = (
        laps_airtable["session_key"].astype(str) + "_" +
        laps_airtable["driver_number"].astype(str) + "_" +
        laps_airtable["lap_number"].astype(str)
    )
    laps_airtable = laps_airtable.rename(columns={
        "session_key": "Session Key",
        "meeting_key": "Meeting Key",
        "driver_number": "Driver Number",
        "lap_number": "Lap Number",
        "date_start": "Date Start",
        "duration_sector_1": "Duration Sector 1",
        "duration_sector_2": "Duration Sector 2",
        "duration_sector_3": "Duration Sector 3",
        "i1_speed": "I1 Speed",
        "i2_speed": "I2 Speed",
        "st_speed": "St Speed",
        "is_pit_out_lap": "Is Pit Out Lap",
        "lap_duration": "Lap Duration",
        "lap_duration_formatted": "Lap Duration Formatted",
        "year": "Year",
    })
    laps_airtable = laps_airtable[[c for c in [
        "Lap Key", "Session Key", "Meeting Key", "Driver Number", "Lap Number", "Date Start",
        "Duration Sector 1", "Duration Sector 2", "Duration Sector 3", "I1 Speed", "I2 Speed",
        "St Speed", "Is Pit Out Lap", "Lap Duration", "Lap Duration Formatted", "Year",
    ] if c in laps_airtable.columns]]
else:
    laps_airtable = pd.DataFrame()

# --- Pit Stops ---
if not pits_scoped.empty:
    pits_airtable = pits_scoped.copy()
    pits_airtable["Pit Key"] = (
        pits_airtable["session_key"].astype(str) + "_" +
        pits_airtable["driver_number"].astype(str) + "_" +
        pits_airtable["lap_number"].astype(str)
    )
    pits_airtable = pits_airtable.rename(columns={
        "session_key": "Session Key",
        "meeting_key": "Meeting Key",
        "driver_number": "Driver Number",
        "lap_number": "Lap Number",
        "date": "Date",
        "pit_duration": "Pit Duration",
    })
    pits_airtable["Year"] = pits_airtable["Session Key"].map(sessions_year_lookup)
    pits_airtable = pits_airtable[[c for c in [
        "Pit Key", "Session Key", "Meeting Key", "Driver Number", "Lap Number", "Date",
        "Pit Duration", "Year",
    ] if c in pits_airtable.columns]]
else:
    pits_airtable = pd.DataFrame()

print("Shaped row counts:")
for name, df_ in [("Meetings", meetings_airtable), ("Sessions", sessions_airtable),
                   ("Drivers", drivers_airtable), ("Session Results", results_airtable),
                   ("Starting Grid", grid_airtable), ("Laps", laps_airtable),
                   ("Pit Stops", pits_airtable)]:
    print(f"  {name}: {len(df_)}")


## 8. Exploratory SQL Joins (across all years)

Same style of query as the original notebooks' "Joining Data Required for Dashboard"
section, generalized so `round` and standings are computed **per year** instead of being
hardcoded to a single season.


In [ ]:
driver_standings_query = """
WITH grand_prix_rounds AS (
    SELECT
        ra.meeting_key,
        ra.year,
        md.meeting_name,
        MIN(ra.date_start) AS grand_prix_date,
        ROW_NUMBER() OVER (PARTITION BY ra.year ORDER BY MIN(ra.date_start)) AS round
    FROM race_sessions_all AS ra
    LEFT JOIN meetings_df AS md
      ON md.meeting_key = ra.meeting_key
    WHERE ra.session_type = 'Race'
    GROUP BY ra.meeting_key, ra.year, md.meeting_name
)

SELECT
    gr.round,
    md.meeting_name AS grand_prix,
    sg.driver_number,
    dr.name_acronym,
    dr.last_name,
    dr.team_name,
    sg.position AS grid_position,
    rs.position AS final_position,
    rs.points,
    ra.year,
    ra.session_name
FROM race_sessions_all AS ra
LEFT JOIN grand_prix_rounds AS gr
  ON gr.meeting_key = ra.meeting_key AND gr.year = ra.year
LEFT JOIN meetings_df AS md
  ON md.meeting_key = ra.meeting_key
LEFT JOIN starting_grid_df AS sg
  ON sg.session_key = ra.session_key
LEFT JOIN session_results_df AS rs
  ON rs.session_key = ra.session_key AND rs.driver_number = sg.driver_number
LEFT JOIN drivers_df AS dr
  ON dr.driver_number = sg.driver_number AND dr.session_key = ra.session_key
ORDER BY ra.year, gr.round
"""

driver_standings = pysqldf(driver_standings_query)
driver_standings.head()


In [ ]:
laps_with_sessions_query = """
SELECT
    ra.year,
    md.meeting_name,
    md.meeting_key,
    ld.driver_number,
    dr.full_name,
    ld.lap_number,
    ld.is_pit_out_lap,
    ld.lap_duration,
    ld.lap_duration_formatted
FROM race_sessions_all AS ra
JOIN meetings_df AS md
  ON md.meeting_key = ra.meeting_key
JOIN laps_df AS ld
  ON ld.session_key = ra.session_key
JOIN drivers_df AS dr
  ON dr.driver_number = ld.driver_number AND dr.session_key = ld.session_key
"""

laps_with_sessions = pysqldf(laps_with_sessions_query) if not laps_df.empty else pd.DataFrame()
laps_with_sessions.head()


## 9. Push Data to Airtable

Upserts each shaped table into its matching Airtable table, keyed on the primary field
set up in Step 2 — safe to re-run (existing rows get updated, new rows get added).


In [ ]:
if PUSH_TO_AIRTABLE:
    push_dataframe_to_airtable(meetings_airtable, "meetings")
    push_dataframe_to_airtable(sessions_airtable, "sessions")
    push_dataframe_to_airtable(drivers_airtable, "drivers")
    push_dataframe_to_airtable(results_airtable, "session_results")
    push_dataframe_to_airtable(grid_airtable, "starting_grid")

    if PUSH_LAPS_TO_AIRTABLE:
        push_dataframe_to_airtable(laps_airtable, "laps")
    else:
        print("Skipping Laps push (PUSH_LAPS_TO_AIRTABLE = False).")

    if PUSH_PITS_TO_AIRTABLE:
        push_dataframe_to_airtable(pits_airtable, "pit_stops")
    else:
        print("Skipping Pit Stops push (PUSH_PITS_TO_AIRTABLE = False).")
else:
    print("PUSH_TO_AIRTABLE is False — nothing was sent to Airtable.")


## 10. Summary

In [ ]:
print("Seasons in scope:", sorted(sessions_scoped["year"].dropna().unique().tolist()))
print(f"Meetings: {len(meetings_airtable)}")
print(f"Sessions: {len(sessions_airtable)}")
print(f"Driver-session rows: {len(drivers_airtable)}")
print(f"Session results: {len(results_airtable)}")
print(f"Starting grid rows: {len(grid_airtable)}")
print(f"Laps: {len(laps_airtable)}")
print(f"Pit stops: {len(pits_airtable)}")
